# 🔍 AML Smurfing Detection Engine
**System:** Anti-Money Laundering Detection | Neo4j Aura Enterprise 5.27  
**Module Coverage:** Module 1 → Module 7  
**Detects:** Scatter-Gather (1 → many → 1) and Gather-Scatter (many → 1 → many)

---

### Architecture
```
Module 1  : Seed Selection        (reuse fan-out/fan-in hub candidates)
Module 2A : Scatter Stage          (source → N intermediaries)
Module 2B : Gather Stage           (N intermediaries → destination)
Module 3  : Chain Matcher          (intersect scatter/gather intermediary sets)
Module 4  : Merge & Remove Duplicates
Module 5  : Feature Engineering    (amount conservation, chain direction)
Module 6  : Risk Scoring
Module 7  : Validation & Evaluation
```

### Timing Design Decision
The chain matcher does **not** enforce strict per-transaction ordering between
the scatter and gather stages (real-world timestamps are noisy and intermediaries
forward funds at different speeds). Instead:

1. The **entire chain** (earliest scatter txn → latest gather txn) must fall within
   `MAX_CHAIN_SPAN_SECONDS` (default: 7 days).
2. **Direction is inferred, not filtered** — we compare median scatter timestamp
   vs median gather timestamp to label the chain as `scatter_gather` or
   `gather_scatter`. This becomes a feature for scoring, not a hard gate that
   discards data. This improves recall while still capturing directionality.

---
## ⚙️ Setup
Mount Google Drive and install required packages.

In [20]:
from google.colab import drive
drive.mount('/content/drive')

!pip install neo4j pandas scikit-learn --quiet

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
## 🔌 Connect to Neo4j
Establish the driver connection using credentials stored in Colab Secrets.

In [21]:
import pandas as pd
import numpy as np
import time
import os
from datetime import datetime, timezone
from google.colab import userdata
from neo4j import GraphDatabase

URI      = userdata.get('NEO4J_URI').strip()
USERNAME = userdata.get('NEO4J_USERNAME').strip()
PASSWORD = userdata.get('NEO4J_PASSWORD').strip()

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
driver.verify_connectivity()
print('✓ Connected to Neo4j AuraDB')


def to_epoch(ts):
    """Convert a Neo4j DateTime object or ISO string to a Unix timestamp."""
    if ts is None:
        return None
    if hasattr(ts, 'to_native'):
        return ts.to_native().replace(tzinfo=timezone.utc).timestamp()
    if isinstance(ts, str):
        return datetime.fromisoformat(ts).timestamp()
    return float(ts)


def run_query_on_batch(session, query, batch, label, **extra_params):
    """Execute one Cypher query for one seed batch, with isolated error handling."""
    try:
        result = session.run(query, seed_ids=batch, **extra_params)
        return result.data()
    except Exception as e:
        print(f'  [WARN] {label} | batch failed → {e}')
        return []


def make_batches(id_list, batch_size=200):
    return [id_list[i:i + batch_size] for i in range(0, len(id_list), batch_size)]


print('Shared helpers defined ✓')

✓ Connected to Neo4j AuraDB
Shared helpers defined ✓


---
## MODULE 1 — Seed Selection
---

### Cell 1 : Load or Recompute Fan-out / Fan-in Hub Candidates
Smurfing candidates are simply the **same accounts** that qualified as fan-out or
fan-in hubs in the Layering module — a scatter-gather chain is, structurally, a
fan-out event whose receivers substantially overlap with a fan-in event's senders.

If you already have `layering_df` in memory from running the Layering notebook in
the same session, this cell reuses it directly. Otherwise it recomputes hub
candidates fresh from Neo4j.

In [22]:
REUSE_LAYERING_RESULTS = 'layering_df' in globals() and not layering_df.empty

if REUSE_LAYERING_RESULTS:
    print('Reusing layering_df from an earlier module run in this session.')

    scatter_source_events = layering_df[layering_df['pattern_type'] == 'fan_out'].copy()
    gather_dest_events    = layering_df[layering_df['pattern_type'] == 'fan_in'].copy()

    print(f'  Fan-out (scatter source) events : {len(scatter_source_events)}')
    print(f'  Fan-in  (gather destination) events : {len(gather_dest_events)}')

else:
    print('layering_df not found in memory — recomputing hub candidates from Neo4j.')

    ACCOUNT_DEGREE_QUERY = """
    MATCH (a:Account)
    OPTIONAL MATCH (a)-[out:TRANSACTION]->(receiver:Account)
    WITH a, count(DISTINCT receiver) AS out_degree
    OPTIONAL MATCH (sender:Account)-[inc:TRANSACTION]->(a)
    WITH a, out_degree, count(DISTINCT sender) AS in_degree
    RETURN a.account_id AS account_id, out_degree, in_degree
    """

    with driver.session() as session:
        degree_stats = session.run(ACCOUNT_DEGREE_QUERY).data()

    degree_df = pd.DataFrame(degree_stats)

    MIN_FANOUT = 3
    MIN_FANIN  = 3

    scatter_seed_ids = degree_df.loc[degree_df['out_degree'] >= MIN_FANOUT, 'account_id'].tolist()
    gather_seed_ids  = degree_df.loc[degree_df['in_degree']  >= MIN_FANIN,  'account_id'].tolist()

    print(f'  Scatter candidate accounts : {len(scatter_seed_ids)}')
    print(f'  Gather  candidate accounts : {len(gather_seed_ids)}')

layering_df not found in memory — recomputing hub candidates from Neo4j.
  Scatter candidate accounts : 349
  Gather  candidate accounts : 399


### Cell 2 : Build Scatter and Gather Seed Batches
If we recomputed candidates fresh (no `layering_df` available), we still need to
run the scatter/gather Cypher stages below. This cell prepares batches only for
that fallback path — skipped automatically if we're reusing `layering_df`.

In [23]:
if not REUSE_LAYERING_RESULTS:
    scatter_batches = make_batches(scatter_seed_ids)
    gather_batches  = make_batches(gather_seed_ids)

    print(f'Scatter batches : {len(scatter_batches)}')
    print(f'Gather  batches : {len(gather_batches)}')
else:
    print('Skipped — reusing pre-computed layering_df events, no fresh Cypher sweep needed.')

Scatter batches : 2
Gather  batches : 2


---
## MODULE 2A / 2B — Scatter & Gather Stages
*(only runs if layering_df was not available — otherwise skip to Module 3)*
---

### Cell 1 : Scatter Stage Query (Fan-out, Time-Windowed)
Identical logic to the Layering module's fan-out detector: pulls every outgoing
transaction per candidate source, to be windowed in Python.

In [24]:
SCATTER_QUERY = """
UNWIND $seed_ids AS origin_id

MATCH (a:Account {account_id: origin_id})-[t:TRANSACTION]->(b:Account)

WITH a, collect({
    receiver   : b.account_id,
    amount     : t.amount_paid,
    timestamp  : t.timestamp,
    laundering : t.is_laundering
}) AS out_txns

RETURN a.account_id AS hub_account, out_txns
"""

print('Scatter stage query defined ✓')

Scatter stage query defined ✓


### Cell 2 : Gather Stage Query (Fan-in, Time-Windowed)
Mirror of the scatter query — pulls every incoming transaction per candidate
destination.

In [25]:
GATHER_QUERY = """
UNWIND $seed_ids AS origin_id

MATCH (b:Account)-[t:TRANSACTION]->(a:Account {account_id: origin_id})

WITH a, collect({
    sender     : b.account_id,
    amount     : t.amount_paid,
    timestamp  : t.timestamp,
    laundering : t.is_laundering
}) AS in_txns

RETURN a.account_id AS hub_account, in_txns
"""

print('Gather stage query defined ✓')

Gather stage query defined ✓


### Cell 3 : Run Scatter & Gather Sweeps (Fallback Path Only)
Runs both Cypher sweeps and converts them into windowed events, using the same
windowing helper approach as the Layering module. Skipped entirely if we're
reusing `layering_df`.

In [26]:
WINDOW_SECONDS = 86400   # 24h grouping window for a single scatter/gather event
MIN_GROUP_SIZE = 3       # minimum distinct counterparties to count as an event


def extract_windowed_events(raw_records, side, window_seconds=WINDOW_SECONDS, min_group=MIN_GROUP_SIZE):
    """
    Generic windowing extractor for scatter ('out') or gather ('in') records.
    side: 'out' expects out_txns/receiver keys, 'in' expects in_txns/sender keys.
    """
    txn_key   = 'out_txns' if side == 'out' else 'in_txns'
    party_key = 'receiver' if side == 'out' else 'sender'

    events = []
    for record in raw_records:
        hub  = record['hub_account']
        txns = record.get(txn_key) or []

        parsed = []
        for tx in txns:
            epoch = to_epoch(tx.get('timestamp'))
            if epoch is not None:
                parsed.append({**tx, 'epoch': epoch})

        if not parsed:
            continue

        parsed.sort(key=lambda x: x['epoch'])
        window_start = parsed[0]['epoch']
        windowed = [tx for tx in parsed if tx['epoch'] - window_start <= window_seconds]

        counterparties = {tx[party_key] for tx in windowed}

        if len(counterparties) >= min_group:
            events.append({
                'hub_account'   : hub,
                'counterparties': sorted(counterparties),
                'amounts'       : [tx['amount'] for tx in windowed],
                'timestamps'    : [tx['timestamp'] for tx in windowed],
                'is_laundering' : [tx['laundering'] for tx in windowed],
            })

    return events


if not REUSE_LAYERING_RESULTS:

    raw_scatter = []
    raw_gather  = []

    print('Running scatter sweep...')
    start = time.time()
    with driver.session() as session:
        for i, batch in enumerate(scatter_batches):
            raw_scatter.extend(run_query_on_batch(session, SCATTER_QUERY, batch, 'scatter'))
    print(f'  Scatter sweep done in {time.time()-start:.1f}s — {len(raw_scatter)} hub records')

    print('Running gather sweep...')
    start = time.time()
    with driver.session() as session:
        for i, batch in enumerate(gather_batches):
            raw_gather.extend(run_query_on_batch(session, GATHER_QUERY, batch, 'gather'))
    print(f'  Gather sweep done in {time.time()-start:.1f}s — {len(raw_gather)} hub records')

    scatter_events = extract_windowed_events(raw_scatter, side='out')
    gather_events  = extract_windowed_events(raw_gather,  side='in')

    scatter_source_events = pd.DataFrame(scatter_events).rename(columns={'hub_account': 'hub_account'})
    gather_dest_events    = pd.DataFrame(gather_events).rename(columns={'hub_account': 'hub_account'})

    print(f'\nScatter events : {len(scatter_source_events)}')
    print(f'Gather  events : {len(gather_dest_events)}')

else:
    print('Skipped — using scatter_source_events / gather_dest_events from layering_df already.')

Running scatter sweep...
  Scatter sweep done in 48.8s — 349 hub records
Running gather sweep...
  Gather sweep done in 1.2s — 399 hub records

Scatter events : 234
Gather  events : 168


---
## MODULE 3 — Chain Matcher
*(the core new logic — pure Python, no Cypher)*
---

### Cell 1 : Configure Chain-Matching Parameters
Per the timing design decision: **no strict per-transaction ordering**. Instead,
the whole chain is bounded by `MAX_CHAIN_SPAN_SECONDS`, and direction is inferred
from median timestamps rather than used as a filter.

In [27]:
MIN_OVERLAP_RATIO      = 0.5        # ≥ 50% of intermediaries must appear in both stages
MAX_CHAIN_SPAN_SECONDS = 7 * 86400  # entire chain must fit within 7 days

print(f'MIN_OVERLAP_RATIO      : {MIN_OVERLAP_RATIO}')
print(f'MAX_CHAIN_SPAN_SECONDS : {MAX_CHAIN_SPAN_SECONDS} ({MAX_CHAIN_SPAN_SECONDS/86400:.0f} days)')

MIN_OVERLAP_RATIO      : 0.5
MAX_CHAIN_SPAN_SECONDS : 604800 (7 days)


### Cell 2 : Build the Chain Matcher
For every scatter event and every gather event, compute the overlap between
their intermediary sets. If the overlap ratio clears `MIN_OVERLAP_RATIO` **and**
the combined chain span fits within `MAX_CHAIN_SPAN_SECONDS`, it's recorded as a
candidate chain. Direction (`scatter_gather` vs `gather_scatter`) is assigned by
comparing median timestamps — a soft signal, not a filter, so noisy or
simultaneous chains are still captured rather than dropped.

In [28]:
def match_chains(scatter_df, gather_df, min_overlap=MIN_OVERLAP_RATIO, max_span=MAX_CHAIN_SPAN_SECONDS):
    """
    Cross-match every scatter event against every gather event by intermediary
    overlap. This is an O(n × m) comparison — fine at the scale of hub-level
    events (typically hundreds, not millions), since the expensive graph work
    already happened in Cypher.

    Returns a list of dicts, one per matched chain.
    """
    chains = []

    if scatter_df.empty or gather_df.empty:
        return chains

    for _, scatter_row in scatter_df.iterrows():
        scatter_set = set(scatter_row['counterparties'])
        if not scatter_set:
            continue

        scatter_epochs = [to_epoch(t) for t in (scatter_row.get('timestamps') or [])]
        scatter_epochs = [e for e in scatter_epochs if e is not None]
        if not scatter_epochs:
            continue

        for _, gather_row in gather_df.iterrows():
            gather_set = set(gather_row['counterparties'])
            if not gather_set:
                continue

            # A source cannot smurf money to itself as the collection point
            if scatter_row['hub_account'] == gather_row['hub_account']:
                continue

            overlap = scatter_set & gather_set
            smaller_set_size = min(len(scatter_set), len(gather_set))
            overlap_ratio = len(overlap) / smaller_set_size if smaller_set_size else 0.0

            if overlap_ratio < min_overlap:
                continue

            gather_epochs = [to_epoch(t) for t in (gather_row.get('timestamps') or [])]
            gather_epochs = [e for e in gather_epochs if e is not None]
            if not gather_epochs:
                continue

            # Whole-chain span check (loose bound, not strict ordering)
            all_epochs = scatter_epochs + gather_epochs
            chain_span = max(all_epochs) - min(all_epochs)

            if chain_span > max_span:
                continue

            # Direction inferred from median timestamps — a soft signal used
            # for scoring/labeling, NOT a filter that would discard the chain
            median_scatter = float(np.median(scatter_epochs))
            median_gather  = float(np.median(gather_epochs))
            chain_direction = 'scatter_gather' if median_gather >= median_scatter else 'gather_scatter'

            chains.append({
                'source_account'      : scatter_row['hub_account'],
                'destination_account' : gather_row['hub_account'],
                'intermediaries'      : sorted(overlap),
                'intermediary_count'  : len(overlap),
                'overlap_ratio'       : round(overlap_ratio, 4),
                'chain_direction'     : chain_direction,
                'chain_span_seconds'  : chain_span,
                'scatter_amounts'     : scatter_row['amounts'],
                'gather_amounts'      : gather_row['amounts'],
                'scatter_timestamps'  : scatter_row['timestamps'],
                'gather_timestamps'   : gather_row['timestamps'],
                'scatter_laundering'  : scatter_row['is_laundering'],
                'gather_laundering'   : gather_row['is_laundering'],
            })

    return chains


print('Chain matcher defined ✓')
print('Running chain matcher...')

start = time.time()
matched_chains = match_chains(scatter_source_events, gather_dest_events)
print(f'Chain matching complete in {time.time()-start:.1f}s')
print(f'Matched chains found : {len(matched_chains)}')

Chain matcher defined ✓
Running chain matcher...
Chain matching complete in 2.5s
Matched chains found : 24


---
## MODULE 4 — Merge & Remove Duplicates
---

### Cell 1 : Convert to DataFrame & Deduplicate
Fingerprints by `(source, destination, sorted intermediary set)` — the same
source/destination pair could otherwise appear more than once if their
intermediary sets overlap only partially across multiple 24h windows.

In [29]:
if matched_chains:
    smurfing_raw_df = pd.DataFrame(matched_chains)

    def make_fingerprint(row):
        inter_key = '|'.join(row['intermediaries'])
        return f"{row['source_account']}::{row['destination_account']}::{inter_key}"

    smurfing_raw_df['fingerprint'] = smurfing_raw_df.apply(make_fingerprint, axis=1)

    rows_before = len(smurfing_raw_df)
    smurfing_raw_df = smurfing_raw_df.drop_duplicates(
        subset=['fingerprint'], keep='first'
    ).reset_index(drop=True)
    rows_after = len(smurfing_raw_df)

    print(f'Chains before dedup : {rows_before}')
    print(f'Chains after  dedup : {rows_after}')
    print(f'Duplicates removed  : {rows_before - rows_after}')
    print()
    print('Breakdown by direction:')
    print(smurfing_raw_df['chain_direction'].value_counts().to_string())

else:
    smurfing_raw_df = pd.DataFrame(columns=[
        'source_account', 'destination_account', 'intermediaries', 'intermediary_count',
        'overlap_ratio', 'chain_direction', 'chain_span_seconds',
        'scatter_amounts', 'gather_amounts', 'scatter_timestamps', 'gather_timestamps',
        'scatter_laundering', 'gather_laundering', 'fingerprint'
    ])
    print('No smurfing chains detected.')

smurfing_raw_df.head()

Chains before dedup : 24
Chains after  dedup : 24
Duplicates removed  : 0

Breakdown by direction:
chain_direction
scatter_gather    20
gather_scatter     4


,source_account,destination_account,intermediaries,intermediary_count,overlap_ratio,chain_direction,chain_span_seconds,scatter_amounts,gather_amounts,scatter_timestamps,gather_timestamps,scatter_laundering,gather_laundering,fingerprint
0,119_811C597B0,48309_811C599A0,"[119_811C597B0, 48309_811C599A0]",2,0.6667,gather_scatter,84360.0,"[34254.65, 1477.92, 102105.84]","[34254.65, 662658.0, 48936.67, 27247.23, 771.9...","[2022-09-01T00:04:00.000000000+00:00, 2022-09-...","[2022-09-01T00:04:00.000000000+00:00, 2022-09-...","[1, 0, 0]","[1, 0, 0, 1, 0, 0, 0, 1]",119_811C597B0::48309_811C599A0::119_811C597B0|...
1,48309_811C599A0,119_811C597B0,"[119_811C597B0, 48309_811C599A0]",2,0.6667,gather_scatter,128760.0,"[662658.0, 90922.81, 391227.05, 48972.89]","[1147.24, 303.75, 1270758.44, 6883941.92, 5327...","[2022-09-01T00:25:00.000000000+00:00, 2022-09-...","[2022-09-01T15:32:00.000000000+00:00, 2022-09-...","[0, 0, 0, 1]","[0, 0, 0, 0, 0, 0, 1, 0, 0, 1]",48309_811C599A0::119_811C597B0::119_811C597B0|...
2,70_100428A08,150240_812D22980,"[150240_812D22980, 48211_811EC3910]",2,0.6667,scatter_gather,86220.0,"[10859.14, 5329.66, 2579.05, 49521.82, 68.29, ...","[1169.66, 10095.18, 155.93, 120.43, 426.75]","[2022-09-01T07:17:00.000000000+00:00, 2022-09-...","[2022-09-01T07:26:00.000000000+00:00, 2022-09-...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0]",70_100428A08::150240_812D22980::150240_812D229...
3,213952_808338D40,25534_8095B6AF0,"[213952_808338D40, 25534_8095B6AF0]",2,0.6667,scatter_gather,57180.0,"[20649.41, 1449610.43, 13753.42, 5436.54, 1249...","[20649.41, 219428.87, 3888.25]","[2022-09-01T07:57:00.000000000+00:00, 2022-09-...","[2022-09-01T07:57:00.000000000+00:00, 2022-09-...","[1, 1, 0, 0, 0]","[1, 0, 0]",213952_808338D40::25534_8095B6AF0::213952_8083...
4,21745_803366440,112931_805D4C850,"[12719_80227FC50, 18097_803144A90, 1_800B047D0...",6,0.7500,scatter_gather,295020.0,"[9479.43, 43038.53, 5483.92, 13312.15, 11360.5...","[753279.46, 564572.32, 147328.48, 977697.65, 1...","[2022-09-02T14:20:00.000000000+00:00, 2022-09-...","[2022-09-05T01:21:00.000000000+00:00, 2022-09-...","[1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1]","[1, 1, 0, 1, 1, 1, 1, 1, 1, 0]",21745_803366440::112931_805D4C850::12719_80227...


---
## MODULE 5 — Feature Engineering
---

### Cell 1 : Define Feature Engineering Function
The standout feature here is **amount conservation ratio** — real smurfing
preserves value (minus a small skim), so `gather_total / scatter_total` should
sit close to 1.0. Large deviations suggest either noise or a cut being taken
by the launderer.

In [30]:
def engineer_smurfing_features(row: pd.Series) -> pd.Series:
    """
    Compute derived features for a single matched scatter/gather chain.

    Structural : intermediary_count, overlap_ratio (already present)
    Amount     : scatter_total, gather_total, amount_conservation_ratio
    Temporal   : chain_span_seconds (already present), scatter_to_gather_delay
    Label      : any_laundering (binary flag only, no ratio — consistent with
                 the layering module's convention)
    """
    scatter_amounts = [a for a in (row.get('scatter_amounts') or []) if a is not None]
    gather_amounts  = [a for a in (row.get('gather_amounts')  or []) if a is not None]

    scatter_total = sum(scatter_amounts)
    gather_total  = sum(gather_amounts)

    # Amount conservation: how much of the scattered value reappears at gather.
    # Capped so a single tiny scatter_total doesn't blow the ratio up unreasonably.
    if scatter_total > 0:
        amount_conservation_ratio = min(gather_total / scatter_total, 3.0)
    else:
        amount_conservation_ratio = 0.0

    # Scatter-to-gather delay: median gather time minus median scatter time.
    # Can be negative for gather_scatter chains — that's expected, not an error.
    scatter_epochs = [to_epoch(t) for t in (row.get('scatter_timestamps') or [])]
    gather_epochs  = [to_epoch(t) for t in (row.get('gather_timestamps')  or [])]
    scatter_epochs = [e for e in scatter_epochs if e is not None]
    gather_epochs  = [e for e in gather_epochs  if e is not None]

    if scatter_epochs and gather_epochs:
        scatter_to_gather_delay = float(np.median(gather_epochs) - np.median(scatter_epochs))
    else:
        scatter_to_gather_delay = 0.0

    # Label: binary flag only (no laundering_ratio, consistent with layering module)
    scatter_labels = [bool(l) for l in (row.get('scatter_laundering') or []) if l is not None]
    gather_labels  = [bool(l) for l in (row.get('gather_laundering')  or []) if l is not None]
    any_laundering = int((sum(scatter_labels) + sum(gather_labels)) > 0)

    return pd.Series({
        'scatter_total_amount'      : scatter_total,
        'gather_total_amount'       : gather_total,
        'amount_conservation_ratio' : round(amount_conservation_ratio, 4),
        'scatter_to_gather_delay'   : scatter_to_gather_delay,
        'any_laundering'            : any_laundering,
    })


print('Feature engineering function defined ✓')

Feature engineering function defined ✓


### Cell 2 : Apply Feature Engineering
Applies the function row-wise and joins the resulting columns onto the base
chain DataFrame to produce `enriched_df`.

In [31]:
if not smurfing_raw_df.empty:

    print('Applying feature engineering...')

    feature_df = smurfing_raw_df.apply(engineer_smurfing_features, axis=1)

    enriched_df = pd.concat(
        [smurfing_raw_df.reset_index(drop=True), feature_df.reset_index(drop=True)],
        axis=1
    )

    print(f'Feature engineering complete — {len(enriched_df)} chains enriched')
    print(f'Total columns : {len(enriched_df.columns)}')

    enriched_df[[
        'source_account', 'destination_account', 'chain_direction',
        'intermediary_count', 'overlap_ratio', 'amount_conservation_ratio',
        'any_laundering'
    ]].head()

else:
    enriched_df = smurfing_raw_df.copy()
    print('No chains to engineer features for.')

Applying feature engineering...
Feature engineering complete — 24 chains enriched
Total columns : 19


---
## MODULE 6 — Risk Scoring
---

### Cell 1 : Define Risk Scoring Functions
Weights per the architecture: overlap ratio (0.35, the strongest structural
signal that two stages are truly connected), amount conservation (0.25, close
to 1.0 = classic value-preserving smurfing), velocity (0.25, faster chains are
more suspicious), and the binary laundering label (0.15).

In [32]:
def compute_smurfing_risk_score(row: pd.Series) -> float:
    """
    Composite risk score for one smurfing chain. Range: 0.0 – 1.0

    Dimension           Weight  Rationale
    ────────────────────────────────────────────
    overlap_score         0.35   Higher intermediary overlap = stronger structural link
    conservation_score     0.25   Ratio near 1.0 = classic value-preserving smurfing
    velocity_score          0.25   Shorter chain span = faster, more suspicious movement
    label_score              0.15   any_laundering ground-truth flag (binary, not ratio)
    """

    # ── Overlap score (already 0–1) ─────────────────
    overlap_score = float(row.get('overlap_ratio', 0.0))

    # ── Conservation score: peaks at ratio == 1.0, decays as it deviates ──
    ratio = float(row.get('amount_conservation_ratio', 0.0))
    conservation_score = max(0.0, 1.0 - abs(ratio - 1.0))

    # ── Velocity score: shorter chain span = higher suspicion ────────
    # 0s → 1.0  |  MAX_CHAIN_SPAN_SECONDS (7 days) → 0.0
    span = float(row.get('chain_span_seconds', MAX_CHAIN_SPAN_SECONDS))
    velocity_score = max(0.0, 1.0 - (span / MAX_CHAIN_SPAN_SECONDS))

    # ── Label score (binary, not ratio) ─────────────────
    label_score = float(row.get('any_laundering', 0))

    intermediary_score = min(
        row["intermediary_count"] / 10,
        1.0
    )

    score = (
          0.35 * overlap_score
        + 0.30 * conservation_score
        + 0.25 * velocity_score
        + 0.10 * intermediary_score
    )

    return round(min(score, 1.0), 4)


def assign_risk_tier(score: float) -> str:
    if   score >= 0.75: return 'CRITICAL'
    elif score >= 0.50: return 'HIGH'
    elif score >= 0.25: return 'MEDIUM'
    else:               return 'LOW'


print('Risk scoring functions defined ✓')

Risk scoring functions defined ✓


### Cell 2 : Apply Scoring → Produce Final `smurfing_df`
Scores every enriched chain, assigns a tier, and sorts descending by risk
score. `smurfing_df` is the canonical output consumed by the GNN, dashboard,
and SAR report modules.

In [33]:
if not enriched_df.empty:

    print('Computing risk scores...')

    enriched_df['risk_score'] = enriched_df.apply(compute_smurfing_risk_score, axis=1)
    enriched_df['risk_tier']  = enriched_df['risk_score'].apply(assign_risk_tier)

    smurfing_df = enriched_df.sort_values(
        by=['risk_score', 'intermediary_count'],
        ascending=[False, False]
    ).reset_index(drop=True)

    print(f'Risk scoring complete — {len(smurfing_df)} chains scored')
    print()
    print('Risk tier distribution:')
    print(smurfing_df['risk_tier'].value_counts().to_string())

else:
    smurfing_df = enriched_df.copy()
    smurfing_df['risk_score'] = pd.Series(dtype=float)
    smurfing_df['risk_tier']  = pd.Series(dtype=str)
    print('No chains to score.')

smurfing_df[[
    'source_account', 'destination_account', 'chain_direction',
    'risk_score', 'risk_tier', 'overlap_ratio', 'amount_conservation_ratio'
]].head(10)

Computing risk scores...
Risk scoring complete — 24 chains scored

Risk tier distribution:
risk_tier
MEDIUM      12
HIGH        11
CRITICAL     1


,source_account,destination_account,chain_direction,risk_score,risk_tier,overlap_ratio,amount_conservation_ratio
0,13862_8078765D0,1601_80340DCF0,scatter_gather,0.7658,CRITICAL,0.7778,1.0154
1,2591_800AEAC00,22_8000EE420,scatter_gather,0.6412,HIGH,0.6000,1.1001
2,29069_807CD84A0,1_80C707860,gather_scatter,0.6294,HIGH,0.6667,0.5226
3,2591_800AEAC00,1_8001BA930,gather_scatter,0.6105,HIGH,0.6667,0.5323
4,20486_807EF78D0,139793_80EB1AA20,scatter_gather,0.6015,HIGH,0.5000,1.1934
5,11405_800EE6B70,1024_80098A3D0,scatter_gather,0.5906,HIGH,0.7500,0.4092
6,19329_805A91C10,23842_8017D32A0,scatter_gather,0.5882,HIGH,0.5000,0.9144
7,5_807E7A050,1547_801C95950,scatter_gather,0.5527,HIGH,0.6667,0.5802
8,15040_8111CA220,10_80CCC1960,scatter_gather,0.5434,HIGH,0.7500,0.2696
9,213952_808338D40,25534_8095B6AF0,scatter_gather,0.5284,HIGH,0.6667,0.1624


---
## MODULE 7 — Validation & Evaluation
---

### Cell 1 : Classification Report & Confusion Matrix
Uses ground-truth `any_laundering` as the positive class. `labels=` is passed
explicitly so the code doesn't crash if only one class is present in a given run.

In [34]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print('=' * 65)
print('MODULE 7 — Validation & Evaluation')
print('=' * 65)

if smurfing_df.empty:
    print('No chains to evaluate.')

else:
    y_true = smurfing_df['any_laundering'].astype(int)

    DETECTION_THRESHOLD = 0.25
    y_pred = (smurfing_df['risk_score'] >= DETECTION_THRESHOLD).astype(int)

    print(f'\nDetection threshold : {DETECTION_THRESHOLD}')
    print(f'Total chains         : {len(smurfing_df)}')
    print(f'Ground-truth positives (any_laundering=1) : {y_true.sum()}')
    print(f'Predicted  positives (score >= threshold) : {y_pred.sum()}')

    present_classes = sorted(y_true.unique())
    class_names     = {0: 'Clean', 1: 'Suspicious'}
    target_names    = [class_names[c] for c in present_classes]

    print('\nClassification Report:')
    print(classification_report(
        y_true, y_pred,
        labels=present_classes,
        target_names=target_names,
        zero_division=0
    ))

    print('Confusion Matrix (rows = actual, cols = predicted):')
    cm = confusion_matrix(y_true, y_pred, labels=present_classes)
    cm_df = pd.DataFrame(
        cm,
        index=  [f'Actual {n}' for n in target_names],
        columns=[f'Pred {n}'   for n in target_names]
    )
    print(cm_df.to_string())

    if y_true.nunique() > 1:
        auc = roc_auc_score(y_true, smurfing_df['risk_score'])
        print(f'\nROC-AUC Score : {auc:.4f}')
    else:
        unique_class = class_names[y_true.iloc[0]]
        print(f'\nROC-AUC : skipped — all chains are "{unique_class}" (only 1 class present)')

MODULE 7 — Validation & Evaluation

Detection threshold : 0.25
Total chains         : 24
Ground-truth positives (any_laundering=1) : 24
Predicted  positives (score >= threshold) : 24

Classification Report:
              precision    recall  f1-score   support

  Suspicious       1.00      1.00      1.00        24

    accuracy                           1.00        24
   macro avg       1.00      1.00      1.00        24
weighted avg       1.00      1.00      1.00        24

Confusion Matrix (rows = actual, cols = predicted):
                   Pred Suspicious
Actual Suspicious               24

ROC-AUC : skipped — all chains are "Suspicious" (only 1 class present)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


### Cell 2 : Per-Direction Breakdown
Shows whether scatter-gather or gather-scatter chains are more prevalent and
which direction the model catches laundering in more reliably — useful for
tuning thresholds and for GNN feature emphasis.

In [35]:
if not smurfing_df.empty:

    summary = smurfing_df.groupby('chain_direction').agg(
        total_chains        = ('risk_score',          'count'),
        avg_risk_score      = ('risk_score',          'mean'),
        laundering_chains   = ('any_laundering',      'sum'),
        avg_overlap_ratio   = ('overlap_ratio',        'mean'),
        avg_conservation    = ('amount_conservation_ratio', 'mean'),
        critical_count      = ('risk_tier', lambda x: (x == 'CRITICAL').sum()),
        high_count          = ('risk_tier', lambda x: (x == 'HIGH').sum()),
    ).reset_index()

    summary['laundering_hit_rate'] = (
        summary['laundering_chains'] / summary['total_chains']
    ).round(4)

    print('Per-direction summary:')
    print(summary.to_string(index=False))

    print(f'\nOverall laundering hit rate : {smurfing_df["any_laundering"].mean():.2%}')
    print(f'Mean risk score             : {smurfing_df["risk_score"].mean():.4f}')
    print(f'CRITICAL tier chains        : {(smurfing_df["risk_tier"] == "CRITICAL").sum()}')

Per-direction summary:
chain_direction  total_chains  avg_risk_score  laundering_chains  avg_overlap_ratio  avg_conservation  critical_count  high_count  laundering_hit_rate
 gather_scatter             4        0.539625                4.0            0.66670          1.763725               0           2                  1.0
 scatter_gather            20        0.506730               20.0            0.65481          1.061205               1           9                  1.0

Overall laundering hit rate : 100.00%
Mean risk score             : 0.5122
CRITICAL tier chains        : 1


### Cell 3 : Save Final Outputs
Persists `smurfing_df` and a top-50 high-risk subset to Google Drive for use
by the GNN training, structuring/dormancy modules, and the investigator
dashboard.

In [36]:
# ============================================================
# MODULE 7
# CELL 3
# Save Complete Results & HIGH/CRITICAL Alerts
# ============================================================

OUTPUT_DIR = '/content/drive/MyDrive/AML System/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Create HIGH & CRITICAL alerts
# ------------------------------------------------------------
alerts_df = smurfing_df[
    smurfing_df["risk_tier"].isin(["HIGH", "CRITICAL"])
].copy()

# Sort highest risk first
alerts_df = alerts_df.sort_values(
    by="risk_score",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# Output file paths
# ------------------------------------------------------------
full_output_path = f"{OUTPUT_DIR}/smurfing_detection_results.csv"

alerts_output_path = (
    f"{OUTPUT_DIR}/smurfing_high_critical_alerts.csv"
)

# ------------------------------------------------------------
# Save CSV files
# ------------------------------------------------------------
smurfing_df.to_csv(full_output_path, index=False)
alerts_df.to_csv(alerts_output_path, index=False)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print("=" * 60)
print("Outputs saved to Google Drive")
print("=" * 60)

print(f"✓ Full Results              : {len(smurfing_df)} rows")
print(f"✓ HIGH & CRITICAL Alerts    : {len(alerts_df)} rows")

print(f"\nSaved:")
print(f"  {full_output_path}")
print(f"  {alerts_output_path}")

print("\nsmurfing_df is ready → Next Module")

# Preview alerts
alerts_df.head()

Outputs saved to Google Drive
✓ Full Results              : 24 rows
✓ HIGH & CRITICAL Alerts    : 12 rows

Saved:
  /content/drive/MyDrive/AML System/outputs/smurfing_detection_results.csv
  /content/drive/MyDrive/AML System/outputs/smurfing_high_critical_alerts.csv

smurfing_df is ready → Next Module


,source_account,destination_account,intermediaries,intermediary_count,overlap_ratio,chain_direction,chain_span_seconds,scatter_amounts,gather_amounts,scatter_timestamps,...,scatter_laundering,gather_laundering,fingerprint,scatter_total_amount,gather_total_amount,amount_conservation_ratio,scatter_to_gather_delay,any_laundering,risk_score,risk_tier
0,13862_8078765D0,1601_80340DCF0,"[116_80EB726B0, 15723_80273FB80, 21918_80211A0...",7,0.7778,scatter_gather,294780.0,"[9676.17, 14172.64, 15461.79, 286.54, 5508.41,...","[10861.49, 20090.35, 20538.8, 5021.92, 476.04,...","[2022-09-08T23:49:00.000000000+00:00, 2022-09-...",...,"[1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]","[1, 1, 1, 1, 0, 1, 1, 1, 1]",13862_8078765D0::1601_80340DCF0::116_80EB726B0...,1.055883e+05,1.072134e+05,1.0154,193590.0,1.0,0.7658,CRITICAL
1,2591_800AEAC00,22_8000EE420,"[1665_8006A4A30, 1_8001E6840, 23_80027A2B0]",3,0.6000,scatter_gather,287340.0,"[19874.09, 16960.55, 16439.17, 14029.19, 4428....","[15526.31, 4411.66, 13322.2, 35285.8, 261675.0...","[2022-09-08T08:02:00.000000000+00:00, 2022-09-...",...,"[1, 0, 1, 0, 1, 0, 1, 0, 1, 0]","[1, 1, 1, 0, 1, 1]",2591_800AEAC00::22_8000EE420::1665_8006A4A30|1...,4.115450e+05,4.527237e+05,1.1001,261270.0,1.0,0.6412,HIGH
2,29069_807CD84A0,1_80C707860,"[1_80C707860, 29069_807CD84A0]",2,0.6667,gather_scatter,74400.0,"[6560.48, 5598.71, 13196.65, 13718.16, 142.5]","[6560.48, 1728.83, 5210.22, 154.52, 59.29, 678...","[2022-09-07T04:49:00.000000000+00:00, 2022-09-...",...,"[1, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0]",29069_807CD84A0::1_80C707860::1_80C707860|2906...,3.921650e+04,2.049452e+04,0.5226,-34680.0,1.0,0.6294,HIGH
3,2591_800AEAC00,1_8001BA930,"[1_8001BA930, 2591_800AEAC00]",2,0.6667,gather_scatter,127020.0,"[19874.09, 16960.55, 16439.17, 14029.19, 4428....","[4333.96, 165677.23, 32601.96, 16439.17]","[2022-09-08T08:02:00.000000000+00:00, 2022-09-...",...,"[1, 0, 1, 0, 1, 0, 1, 0, 1, 0]","[0, 0, 0, 1]",2591_800AEAC00::1_8001BA930::1_8001BA930|2591_...,4.115450e+05,2.190523e+05,0.5323,-41190.0,1.0,0.6105,HIGH
4,20486_807EF78D0,139793_80EB1AA20,"[121987_80839EF70, 21745_80AA68C00, 32405_811A...",4,0.5000,scatter_gather,255060.0,"[3062.23, 2370.78, 413.18, 835.23, 345.74, 122...","[2372.62, 130471290.55, 613.93, 89709263.78, 5...","[2022-09-04T09:35:00.000000000+00:00, 2022-09-...",...,"[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, ...","[1, 1, 0, 1, 1, 1, 1, 1]",20486_807EF78D0::139793_80EB1AA20::121987_8083...,2.003789e+08,2.391331e+08,1.1934,156570.0,1.0,0.6015,HIGH
